# Lab: Simulating Inflation

AST 3414 - Spring 2026

## Introduction

In this lab, you and a partner will build a numerical simulation of **slow-roll inflation** — the simplest version of inflation that may be viable.

### What You'll Learn

1. Solve the differential equations for the expansion of the early universe for a quadratic **inflaton field** and the **scale factor** during inflation.
2. Calculate slow-roll parameters and identify when inflation ends.Verify that inflation produces the required ~60 e-folds of accelerated expansion.
3. Extend your simulation to model **eternal inflation**, in which quantum fluctuations cause different regions of the universe to stop inflating at different times — producing a **multiverse**.

---

This is a Pair Programming Lab - Please work together with all members looking at the same copy of the code on one computer. One member will be the Driver who controls the keyboard. The others will be the Navigator(s) who reviews code and think strategically, giving instructions to the Drive.

Every 20-30 minutes an announcement will be made to switch roles.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp
from IPython.display import HTML

# We work in natural units: reduced Planck mass M_Pl = 1, c = hbar = 1.
M_Pl = 1.0


---

## Part 1: Slow-Roll Inflation in a Single Universe

Inflation is driven by a scalar field $\phi$ with potential energy $V(\phi)$.
The inflaton is coupled to gravity through the **Friedmann equation**, and its own dynamics
are governed by the **Klein-Gordon equation** (a relativistic wave equation taht describes the quantum behavior of scalar fields) in an expanding spacetime.

In a flat Friedmann-Lema\^itre-Robertson-Walker (FLRW) universe, these equations are:

$$
H^2 = \frac{1}{3 M_{\rm Pl}^2} \left[ \frac{1}{2}\dot{\phi}^2 + V(\phi) \right]
\qquad \text{(Friedmann equation)}
$$

$$
\ddot{\phi} + 3H\dot{\phi} + \frac{dV}{d\phi} = 0
\qquad \text{(Klein-Gordon equation)}
$$

where $H = \dot{a}/a$ is the Hubble parameter, $a(t)$ is the scale factor, and $M_{\rm Pl} = \sqrt{\frac{\hbar c}{8 \pi G}}$ is the reduced Planck mass, which is a constant used to make the equations easier to read when using natural units.

During slow-roll inflation, the field rolls slowly enough that $\ddot{\phi} \approx 0$ and
$\frac{1}{2}\dot\phi^2 \ll V(\phi)$. Under these approximations:

$$
H^2 \approx \frac{V(\phi)}{3 M_{\rm Pl}^2}, \qquad 3H\dot{\phi} \approx -\frac{dV}{d\phi}
$$

The quality of these approximations is quantified by the **slow-roll parameters**:

$$
\epsilon = \frac{M_{\rm Pl}^2}{2}\left(\frac{V'}{V}\right)^2, \qquad
\eta = M_{\rm Pl}^2 \frac{V''}{V}
$$

Inflation occurs when $\epsilon < 1$ and $|\eta| < 1$, and *ends* when $\epsilon \approx 1$.

The total amount of expansion is measured in *e-folds*:

$$
N(t) = \ln\!\left(\frac{a(t)}{a(t_i)}\right) = \int_{t_i}^{t} H \, dt'
$$

We need $N \gtrsim 60$ e-folds to solve the horizon and flatness problems.


### Define the Inflaton Potential

We will use one of the simplest inflationary models: the **quadratic potential**:

$$
V(\phi) = \frac{1}{2} m^2 \phi^2
$$

where $m$ is the inflaton mass. For this potential, the slow-roll parameters are:

$$
\epsilon = \eta = \frac{2 M_{\rm Pl}^2}{\phi^2}
$$

so inflation ends at $\phi_{\rm end} = \sqrt{2}\, M_{\rm Pl}$.

**Task:** Complete the two functions below. The first returns $V(\phi)$, the second returns
$dV/d\phi$.

> **Hint:** For the quadratic potential $V = \frac{1}{2}m^2\phi^2$, the derivative is simply $V' = m^2 \phi$.


In [ ]:
# Inflaton mass (in Planck units). This value gives realistic inflationary dynamics.
m = 6e-6  # m / M_Pl ~ 6 x 10^-6

def V(phi):
    # Inflaton potential: V(phi) = (1/2) m^2 phi^2
    # TODO: return the potential energy
    # Your code here (one line):
    pass

def dVdphi(phi):
    # Derivative of the potential: dV/dphi
    # TODO: return the derivative of V with respect to phi
    # Your code here (one line):
    pass

# ---------- Sanity check ----------
# If implemented correctly, V(1.0) should equal 0.5 * m**2 = 1.8e-11
print(f"V(1.0) = {V(1.0):.3e}   (expected: 1.800e-11)")
print(f"dV/dphi(1.0) = {dVdphi(1.0):.3e}   (expected: 3.600e-11)")


### Set Up the Equations of Motion

To solve the system numerically, we rewrite it as a set of **first-order ODEs**. Our state
vector is $\mathbf{y} = [\phi,\; \dot{\phi},\; \ln a]$. (We evolve $\ln a$ instead of $a$ to
avoid numerical overflow during exponential expansion.)

The system is:

$$
\frac{d\phi}{dt} = \dot{\phi}
$$

$$
\frac{d\dot{\phi}}{dt} = -3H\dot{\phi} - \frac{dV}{d\phi}
$$

$$
\frac{d(\ln a)}{dt} = H
$$

where $H$ is computed from the Friedmann equation at each step:

$$
H = \sqrt{\frac{1}{3 M_{\rm Pl}^2}\left(\frac{1}{2}\dot{\phi}^2 + V(\phi)\right)}
$$

**Task:** Complete the function below that returns the right-hand side of this ODE system.

> **Python tip:** `np.sqrt(x)` computes $\sqrt{x}$. The state vector `y` is a list/array
> where `y[0]` = $\phi$, `y[1]` = $\dot\phi$, `y[2]` = $\ln a$.


In [ ]:
def inflation_odes(t, y):
    '''
    Right-hand side of the inflation ODE system.

    Parameters
    ----------
    t : float
        Time (not used explicitly, but required by the ODE solver).
    y : array-like
        State vector [phi, phi_dot, ln_a].

    Returns
    -------
    list of 3 floats
        [d(phi)/dt, d(phi_dot)/dt, d(ln_a)/dt]
    '''
    phi, phi_dot, ln_a = y

    # Step 1: Compute the Hubble parameter H from the Friedmann equation.
    H = ...  # your code here

    # Step 2: Compute d(phi)/dt.
    dphi_dt = phi_dot

    # Step 3: Compute d(phi_dot)/dt from the Klein-Gordon equation.
    dphi_dot_dt = ...  # your code here

    # Step 4: Compute d(ln a)/dt = H.
    dln_a_dt = H

    return [dphi_dt, dphi_dot_dt, dln_a_dt]


### Choose Initial Conditions and Solve

For the quadratic potential, the field value at the start of the last ~60 e-folds of inflation is
approximately:

$$
\phi_i \approx \sqrt{4N + 2}\; M_{\rm Pl} \approx 15.6\; M_{\rm Pl}
$$

We start with $\dot{\phi}_i$ given by the slow-roll approximation:

$$
\dot{\phi}_i \approx -\frac{V'(\phi_i)}{3 H(\phi_i)}
$$

and we set $\ln a_i = 0$ (i.e., $a_i = 1$).

**Task:** Fill in the initial conditions and run the solver. We use an **event function** to stop
integration when the slow-roll parameter $\epsilon$ reaches 1 (inflation ends).


In [ ]:
# --- Initial field value ---
N_target = 65  # aim for a bit more than 60 e-folds
phi_i = np.sqrt(4 * N_target + 2) * M_Pl

# --- Initial field velocity (slow-roll approximation) ---
H_i = ...       # your code here
phi_dot_i = ... # your code here

# --- Initial ln(a) ---
ln_a_i = 0.0

# --- Pack initial conditions ---
y0 = [phi_i, phi_dot_i, ln_a_i]

print(f"Initial conditions:")
print(f"  phi_i     = {phi_i:.4f} M_Pl")
print(f"  phi_dot_i = {phi_dot_i:.4e} M_Pl^2")
print(f"  H_i       = {H_i:.4e} M_Pl")


Now we define an **event function** that triggers when $\epsilon = 1$ (end of inflation), then
solve the system.

> **Note on `solve_ivp`:** This is SciPy's general-purpose ODE integrator. The key arguments are:
> - `fun`: the right-hand side function
> - `t_span`: the time interval `(t_start, t_end)` — we pick a large upper bound
> - `y0`: the initial state vector
> - `events`: a function whose zero-crossing stops integration
> - `max_step`: limits the step size for smooth output
> - `dense_output=True`: lets us evaluate the solution at any time


In [ ]:
def end_of_inflation(t, y):
    '''
    Event function: returns (epsilon - 1).
    When this crosses zero, inflation has ended.
    '''
    phi = y[0]
    epsilon = 0.5 * M_Pl**2 * (dVdphi(phi) / V(phi))**2
    return epsilon - 1.0

end_of_inflation.terminal = True    # stop integration when triggered
end_of_inflation.direction = 1      # trigger when epsilon crosses 1 from below

# --- Solve ---
# The time scale is set by 1/H ~ 1/(m * phi) in Planck times.
# We pick a generous upper bound for the time span.
t_span = (0, 1e8)  # in Planck times

sol = solve_ivp(
    inflation_odes,
    t_span,
    y0,
    events=end_of_inflation,
    max_step=5e4,
    rtol=1e-10,
    atol=1e-12,
    dense_output=True
)

print(f"Integration terminated at t = {sol.t[-1]:.4e} Planck times")
print(f"  Status: {sol.message}")
print(f"  Number of time steps: {len(sol.t)}")


### Activity 1.5 — Visualize the Inflationary Epoch

**Task:** Run the plotting code below (already complete) and answer the discussion questions that
follow.


In [ ]:
# Extract solution
t = sol.t
phi_sol = sol.y[0]
phi_dot_sol = sol.y[1]
ln_a_sol = sol.y[2]

# Derived quantities
H_sol = np.sqrt((0.5 * phi_dot_sol**2 + V(phi_sol)) / (3 * M_Pl**2))
N_sol = ln_a_sol - ln_a_sol[0]  # e-folds since start
epsilon_sol = 0.5 * M_Pl**2 * (dVdphi(phi_sol) / V(phi_sol))**2

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# (a) Inflaton field vs time
axes[0, 0].plot(t, phi_sol, color='tab:blue')
axes[0, 0].set_xlabel('Time  [Planck times]')
axes[0, 0].set_ylabel(r'$\phi$ [$M_{\rm Pl}$]')
axes[0, 0].set_title('Inflaton Field')
axes[0, 0].axhline(np.sqrt(2), color='red', ls='--', alpha=0.5, label=r'$\phi_{\rm end}=\sqrt{2}$')
axes[0, 0].legend()

# (b) e-folds vs time
axes[0, 1].plot(t, N_sol, color='tab:orange', label=r'$N(t)$ (numerical)')
# Overlay a straight line from the origin with slope H_i for comparison
axes[0, 1].plot(t, H_sol[0] * t, 'k--', alpha=0.3, label=r'$H_i \times t$ (pure de Sitter)')
axes[0, 1].set_xlabel('Time  [Planck times]')
axes[0, 1].set_ylabel(r'$N = \ln(a/a_i)$')
axes[0, 1].set_title('Number of e-Folds')
axes[0, 1].axhline(60, color='red', ls='--', alpha=0.5, label='$N = 60$')
axes[0, 1].legend(fontsize=10)

# (c) Scale factor vs time (LOG SCALE) — shows the explosive growth directly
axes[0, 2].semilogy(t, np.exp(N_sol), color='tab:purple', lw=2, label=r'$a(t) = e^{N(t)}$')
# Compare to power-law expansion a ~ t^2 (radiation-dominated, normalized to match at t_i)
t_nonzero = t[t > 0]
a_power = (t_nonzero / t_nonzero[0])**2  # a ~ t^2 for comparison
axes[0, 2].semilogy(t_nonzero, a_power, 'k--', alpha=0.3, label=r'$a \propto t^2$ (power law)')
axes[0, 2].set_xlabel('Time  [Planck times]')
axes[0, 2].set_ylabel(r'$a(t) / a_i$')
axes[0, 2].set_title('Scale Factor (log scale)')
axes[0, 2].legend(fontsize=10)

# (d) Hubble parameter vs time
axes[1, 0].plot(t, H_sol, color='tab:green')
axes[1, 0].set_xlabel('Time  [Planck times]')
axes[1, 0].set_ylabel(r'$H$ [$M_{\rm Pl}$]')
axes[1, 0].set_title('Hubble Parameter')

# (e) Slow-roll parameter epsilon vs time
axes[1, 1].plot(t, epsilon_sol, color='tab:red')
axes[1, 1].set_xlabel('Time  [Planck times]')
axes[1, 1].set_ylabel(r'$\epsilon$')
axes[1, 1].set_title(r'Slow-Roll Parameter $\epsilon$')
axes[1, 1].axhline(1.0, color='black', ls='--', alpha=0.5, label=r'$\epsilon = 1$')
axes[1, 1].set_yscale('log')
axes[1, 1].legend()

# (f) Equation of state w = (KE - PE)/(KE + PE)
KE = 0.5 * phi_dot_sol**2
PE = V(phi_sol)
w_sol = (KE - PE) / (KE + PE)
axes[1, 2].plot(t, w_sol, color='tab:brown', lw=2)
axes[1, 2].axhline(-1, color='black', ls='--', alpha=0.3, label='$w = -1$ (cosmological constant)')
axes[1, 2].axhline(-1/3, color='gray', ls=':', alpha=0.3, label='$w = -1/3$ (accel. threshold)')
axes[1, 2].set_xlabel('Time  [Planck times]')
axes[1, 2].set_ylabel(r'$w = P/\rho$')
axes[1, 2].set_title('Equation of State')
axes[1, 2].set_ylim(-1.05, 0)
axes[1, 2].legend(fontsize=9)

fig.suptitle('Slow-Roll Inflation: Quadratic Potential', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nTotal e-folds of inflation: {N_sol[-1]:.1f}")
print(f"Final field value: phi_end = {phi_sol[-1]:.4f} M_Pl  (expected: {np.sqrt(2):.4f})")
print(f"Hubble parameter: H_i = {H_sol[0]:.4e}, H_end = {H_sol[-1]:.4e}  (ratio: {H_sol[0]/H_sol[-1]:.1f}x)")


### Discussion Questions — Part 1

Answer the following with your partner. Write 2-3 sentences for each.

**Q1.** Look at the plot of $\phi(t)$. Is inflation rolling quickly or slowly for most of the
inflationary epoch? How does this relate to the name "slow-roll"?

*Your answer:*

**Q2.** Look at the e-folds plot. The dashed black line shows what $N(t)$ would look like if $H$
were exactly constant (pure de Sitter expansion). Why does the actual $N(t)$ curve fall below this
line and bend downward? Now look at the scale factor plot $a(t)$ on a log scale. How does the
inflationary expansion compare to a power-law expansion $a \propto t^2$?

*Your answer:*

**Q3.** What happens to the Hubble parameter $H$ during inflation? Is it exactly constant? What does
this tell you about the difference between slow-roll inflation and pure de Sitter expansion
($H =$ const)?

*Your answer:*


---

## Part 2: Eternal Inflation and the Multiverse

### Background: Quantum Fluctuations During Inflation

During inflation, the inflation field is not perfectly classical. Quantum fluctuations constantly
perturb it. In a single Hubble time $\Delta t \sim H^{-1}$, the inflaton in a given Hubble-volume
patch receives a random kick of typical size:

$$
\delta\phi_{\rm quantum} \sim \frac{H}{2\pi}
$$

Meanwhile, the classical slow-roll carries the field downhill by:

$$
\delta\phi_{\rm classical} \sim |\dot\phi| \times H^{-1} \sim \frac{|V'|}{3H^2}
$$

If the quantum kick is *larger* than the classical roll — i.e., if
$H/(2\pi) > |V'|/(3H^2)$ — then some patches of the universe get kicked *back up* the potential
and keep inflating, even while other patches roll to the bottom and thermalize (stop inflating).

This is the regime of **eternal inflation**. The universe as a whole never stops inflating; it
continually spawns "bubble universes" (or "pocket universes") that thermalize, surrounded by
an ever-expanding inflating background. This is the origin of the **multiverse**.

### Multiverse Model

We will simulate this on a **2D grid** of patches, each one representing a Hubble-volume region.
At each discrete time step (representing one Hubble time $\Delta t = H^{-1}$):

1. Each patch that is still inflating evolves: $\phi \to \phi + \delta\phi_{\rm classical} + \delta\phi_{\rm quantum}$.
2. If a patch's field rolls below $\phi_{\rm end} = \sqrt{2}\, M_{\rm Pl}$, it **thermalizes**
   and stops evolving — it becomes a pocket universe.
3. Patches that are still inflating **expand**. In a real simulation, inflating patches would create
   new volume. We model this by having inflating patches "infect" their thermalized neighbors with
   probability $p_{\rm expand}$, resetting those neighbors to the inflating state with a field value
   drawn from the inflating patch.

This is a simplified toy model, but it captures the essential physics: the competition between
classical rolling (which ends inflation) and quantum fluctuations (which sustain it).


### Rescaling for the Eternal Inflation Regime

Before we begin coding Part 2, we need to address an important point about our parameters.

In Part 1, we used the physically motivated inflaton mass $m = 6\times10^{-6}\, M_{\rm Pl}$. For
the $m^2\phi^2$ potential, the eternal inflation condition — $\delta\phi_{\rm quantum} >
\delta\phi_{\rm classical}$ — requires:

$$
\phi > \phi_{\rm eternal} = \sqrt{\frac{4\pi\sqrt{6}}{m}}
$$

For $m = 6\times 10^{-6}$, this gives $\phi_{\rm eternal} \approx 2{,}265\, M_{\rm Pl}$.
Simulating eternal inflation at those field values would require millions of time steps and is
completely impractical.

This is actually a well-known feature of large-field chaotic inflation: eternal inflation occurs
only at super-Planckian field values, deep in the regime where quantum gravity effects (which we
are not modeling) become important.

To make our toy-model simulation work at manageable field values ($\phi_0 \sim 10$), we **rescale
the inflaton mass** to a larger value. This is not physically realistic, but it correctly captures
the *qualitative* dynamics of eternal inflation — the competition between classical rolling and
quantum fluctuations. Think of it as zooming in on the interesting regime.

Run the cell below to redefine the potential with the toy-model mass.

> **Important:** This redefines `V(phi)` and `dVdphi(phi)` globally. If you want to re-run
> Part 1 afterward, you will need to go back and re-run the cell that sets `m = 6e-6`.


In [ ]:
# --- Rescale inflaton mass for Part 2 ---
# With m = 0.5, the eternal inflation threshold is at phi ~ 8 M_Pl,
# so phi_0 = 10 is solidly eternal and phi_0 = 5 is not.
m = 0.5  # toy-model mass for the multiverse simulation

def V(phi):
    return 0.5 * m**2 * phi**2

def dVdphi(phi):
    return m**2 * phi

# Verify: compare quantum vs classical step sizes at phi_0 = 10
phi_test = 10.0
H_test = np.sqrt(V(phi_test) / (3.0 * M_Pl**2))
delta_classical = abs(dVdphi(phi_test)) / (3.0 * H_test**2)
delta_quantum = H_test / (2.0 * np.pi)

print(f"Toy-model mass: m = {m}")
print(f"\nAt phi = {phi_test} M_Pl:")
print(f"  Classical drift per Hubble time: {delta_classical:.4f} M_Pl")
print(f"  Quantum kick (1-sigma):          {delta_quantum:.4f} M_Pl")
print(f"  Ratio (quantum / classical):     {delta_quantum/delta_classical:.2f}")
print(f"  Eternal inflation regime:        {delta_quantum > delta_classical}")

print(f"\nAt phi = 5.0 M_Pl:")
phi_test2 = 5.0
H_test2 = np.sqrt(V(phi_test2) / (3.0 * M_Pl**2))
delta_c2 = abs(dVdphi(phi_test2)) / (3.0 * H_test2**2)
delta_q2 = H_test2 / (2.0 * np.pi)
print(f"  Classical drift per Hubble time: {delta_c2:.4f} M_Pl")
print(f"  Quantum kick (1-sigma):          {delta_q2:.4f} M_Pl")
print(f"  Ratio (quantum / classical):     {delta_q2/delta_c2:.2f}")
print(f"  Eternal inflation regime:        {delta_q2 > delta_c2}")


### Implement the Stochastic Step

Complete the function below that advances a single inflating patch by one Hubble time.

> **Hint:** `np.random.normal(0, 1)` draws a single sample from a standard normal distribution.


In [ ]:
def stochastic_step(phi_current):
    '''
    Advance the inflaton field in one patch by one Hubble time.

    Parameters
    ----------
    phi_current : float
        Current field value in the patch [M_Pl].

    Returns
    -------
    phi_new : float
        Updated field value after one Hubble time.
    thermalized : bool
        True if the field has rolled below phi_end.
    '''
    phi_end = np.sqrt(2) * M_Pl

    # Step 1: Compute the local Hubble parameter (slow-roll approx).
    H = ...  # your code here

    # Step 2: Classical slow-roll displacement in one Hubble time (Delta t = 1/H).
    delta_classical = ...  # your code here

    # Step 3: Quantum fluctuation in one Hubble time.
    delta_quantum = ...  # your code here

    # Step 4: Update phi.
    phi_new = phi_current + delta_classical + delta_quantum

    # Step 5: Check if thermalized.
    thermalized = (phi_new <= phi_end)

    return phi_new, thermalized


### Run the Multiverse Simulation

Read through the simulation loop below. The grid is initialized with all patches inflating at a field value $\phi_0$. At each time step, inflating patches evolve stochastically, and inflating patches can expand into thermalized neighbors.

Notice how `np.roll(array, shift, axis)` is used for accessing neighboring cells on a grid with periodic boundary conditions.


In [ ]:
def run_multiverse(grid_size=100, n_steps=150, phi_0=10.0, p_expand=0.3, seed=42):
    '''
    Simulate eternal inflation on a 2D grid.

    Parameters
    ----------
    grid_size : int
        Number of patches along each axis.
    n_steps : int
        Number of Hubble-time steps to simulate.
    phi_0 : float
        Initial field value in all patches [M_Pl].
    p_expand : float
        Probability that an inflating patch resets a thermalized neighbor.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    history : list of 2D arrays
        The field value grid at each time step.
    inflating_history : list of 2D bool arrays
        Whether each patch is still inflating at each step.
    '''
        np.random.seed(seed)

    phi_grid = np.full((grid_size, grid_size), phi_0)
    inflating = np.ones((grid_size, grid_size), dtype=bool)

    history = [phi_grid.copy()]
    inflating_history = [inflating.copy()]

    for step in range(n_steps):
        new_phi = phi_grid.copy()
        new_inflating = inflating.copy()

        # --- Evolve all currently inflating patches ---
        for i in range(grid_size):
            for j in range(grid_size):
                if inflating[i, j]:
                    # SOLUTION
                    phi_new, thermalized = stochastic_step(phi_grid[i, j])

                    new_phi[i, j] = phi_new
                    if thermalized:
                        new_inflating[i, j] = False

        # --- Expansion: inflating patches can reclaim thermalized neighbors ---
        for i in range(grid_size):
            for j in range(grid_size):
                if new_inflating[i, j]:
                    neighbors = [
                        ((i-1) % grid_size, j),
                        ((i+1) % grid_size, j),
                        (i, (j-1) % grid_size),
                        (i, (j+1) % grid_size),
                    ]
                    for ni, nj in neighbors:
                        if not new_inflating[ni, nj]:
                            if np.random.random() < p_expand:
                                new_inflating[ni, nj] = True
                                new_phi[ni, nj] = new_phi[i, j] + np.random.normal(0, 0.5)

        phi_grid = new_phi
        inflating = new_inflating
        history.append(phi_grid.copy())
        inflating_history.append(inflating.copy())

        frac_inflating = np.mean(inflating)
        if (step + 1) % 25 == 0:
            print(f"Step {step+1}/{n_steps}: {frac_inflating*100:.1f}% of patches still inflating")

    return history, inflating_history


The simulation may take a minute or so. Try the default parameters first, then experiment with different values in the next section.


In [ ]:
# --- Run with default parameters ---
# phi_0 = 10 M_Pl places us in the eternal inflation regime for the m^2 phi^2 potential.
history, inflating_history = run_multiverse(
    grid_size=100,
    n_steps=150,
    phi_0=10.0,
    p_expand=0.3,
    seed=42
)
print("Simulation complete!")


### Visualize the Multiverse

**Task:** Run the visualization code below, which produces:
1. A snapshot of the final state of the grid (inflating vs. thermalized patches).
2. A time series showing the fraction of the universe still inflating.
3. An animation of the multiverse evolving over time.


In [ ]:
# --- Final state snapshot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: inflating (blue) vs thermalized (red)
final_inflating = inflating_history[-1].astype(float)
im1 = axes[0].imshow(final_inflating, cmap='RdBu', vmin=0, vmax=1, origin='lower')
axes[0].set_title('Final State: Inflating (blue) vs. Thermalized (red)')
axes[0].set_xlabel('Grid x')
axes[0].set_ylabel('Grid y')

# Right panel: field value in the final state
im2 = axes[1].imshow(history[-1], cmap='viridis', origin='lower')
axes[1].set_title(r'Final Field Value $\phi$ [$M_{\rm Pl}$]')
axes[1].set_xlabel('Grid x')
axes[1].set_ylabel('Grid y')
plt.colorbar(im2, ax=axes[1], label=r'$\phi$ [$M_{\rm Pl}$]')

plt.tight_layout()
plt.show()


In [ ]:
# --- Inflating fraction vs time ---
frac_inflating = [np.mean(inf) for inf in inflating_history]

plt.figure(figsize=(9, 4))
plt.plot(frac_inflating, 'b-', lw=2)
plt.xlabel('Time Step (Hubble times)')
plt.ylabel('Fraction of Patches Still Inflating')
plt.title('Eternal Inflation: Does Inflation Ever End Globally?')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


In [ ]:
# --- Animation ---
fig_anim, ax_anim = plt.subplots(figsize=(7, 7))
im = ax_anim.imshow(inflating_history[0].astype(float), cmap='RdBu',
                     vmin=0, vmax=1, origin='lower')
title = ax_anim.set_title('Step 0')

def update(frame):
    im.set_data(inflating_history[frame].astype(float))
    title.set_text(f'Step {frame}  |  Inflating: {np.mean(inflating_history[frame])*100:.1f}%')
    return [im, title]

anim = FuncAnimation(fig_anim, update, frames=range(0, len(inflating_history), 2),
                     interval=100, blit=False)
plt.close(fig_anim)  # prevent static display
HTML(anim.to_jshtml())


## Part 3. Explore the Multiverse Parameter Space

Now it is time to experiment! With your partner, vary the simulation parameters and observe how the
multiverse evolves differently.

Run the simulation with at least **three** different parameter combinations from the
suggestions below. For each, record what you observe about the final inflating fraction and the
visual structure of the multiverse.

| Parameter | Default | Try also | Physical meaning |
|-----------|---------|----------|-----------------|
| `phi_0`   | 10.0    | 5.0, 20.0 | Higher $\phi_0$ means higher $H$ and stronger quantum kicks |
| `p_expand`| 0.3     | 0.0, 0.6 | Models the volume growth of inflating regions |
| `m`       | 6e-6    | 3e-6, 2e-5 | Inflaton mass controls the steepness of the potential |
| `grid_size`| 100    | 50, 200  | Spatial resolution (larger = slower) |

> **Important:** If you change `m`, you need to re-run the cell that defines `V(phi)` and
> `dVdphi(phi)` so the new mass takes effect.


In [ ]:
# Experiment 1: Try a different set of parameters
# Example: lower initial field value --- does inflation still persist?
history_exp1, inflating_exp1 = run_multiverse(
    grid_size=100,
    n_steps=150,
    phi_0=5.0,      # <-- try changing this
    p_expand=0.3,
    seed=42
)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.imshow(inflating_exp1[-1].astype(float), cmap='RdBu', vmin=0, vmax=1, origin='lower')
ax.set_title(r'$\phi_0 = 5\, M_{\rm Pl}$: Final State')
plt.tight_layout()
plt.show()
print(f"Final inflating fraction: {np.mean(inflating_exp1[-1])*100:.1f}%")


In [ ]:
# Space for Experiments 2 and 3 --- modify and re-run as needed.
# Try different combinations and discuss with your partner!

# Experiment 2:
# history_exp2, inflating_exp2 = run_multiverse(...)

# Experiment 3:
# history_exp3, inflating_exp3 = run_multiverse(...)


### Discussion Questions — Part 2

**Q4.** In your default simulation, does the inflating fraction ever reach zero? What does this imply about whether inflation can truly "end" globally in this model?

*Your answer:*

**Q5.** What happens when you lower $\phi_0$ to 5.0? Does the multiverse still sustain eternal inflation, or does inflation end everywhere? Why?

*Your answer:*

**Q6.** What is the role of `p_expand` in the simulation? Physically, this represents the fact that inflating regions expand exponentially and create new volume. What happens when you set it to 0?

*Your answer:*

**Q7.** Each thermalized patch in our simulation represents a "pocket universe" — a region where inflation has ended and (in a more realistic model) the Big Bang, nucleosynthesis, and galaxy
formation could proceed. Our own observable universe would be *one such patch*. What does the spatial distribution of thermalized patches in your simulation suggest about the relationship between neighboring pocket universes?

*Your answer:*

**Q8.** The multiverse idea is sometimes criticized as being "unfalsifiable" because we can never observe other pocket universes. Do you think this is a fair criticism? Discuss with your partner whether a theory needs to be directly testable in *every* prediction it makes to be considered scientific.

*Your answer:*


---

## Summary

In this lab, you:

- Implemented the equations of motion for slow-roll inflation and solved them numerically.
- Verified that a quadratic inflaton potential produces ~60+ e-folds of expansion before inflation ends.
- Built a stochastic simulation of eternal inflation on a 2D grid, modeling the competition between
  classical rolling and quantum fluctuations.
- Explored how the multiverse structure depends on the inflaton mass, initial field value, and
  expansion dynamics.

The key takeaway is that inflation, once it starts, may be very difficult to stop *everywhere*.
Quantum fluctuations guarantee that some regions of the universe keep inflating even as others
thermalize. This is the origin of the multiverse — not a speculative add-on, but a natural
consequence of the same physics that solves the horizon and flatness problems.

---

*Lab complete! Make sure both partners' names are on the submission.*
